In [10]:
import io
import copy
import os
import pandas as pd
import sys
import string
import re

os.chdir('/home/labs/davidgo/Collaboration/')

###  import archaic dervied MPRA results

In [2]:
MPRA_results = pd.read_csv(f'humanMPRA/top_candidates/chondrocytes/humanMPRA_annotations_v3.csv', 
                         #usecols=range(0, 34), 
                         header=0)

/tmp/ipykernel_920990/2820908048.py:1: DtypeWarning: Columns (34) have mixed types. Specify dtype option on import or set low_memory=False.
  MPRA_results = pd.read_csv(f'humanMPRA/top_candidates/chondrocytes/humanMPRA_annotations_v3.csv',


### Important

#### WTF always calculated REF - ALT. Therefor I want to get 

#### REF - ALT = Derived - Ancestral = Modern - Ancestral

#### So the varaint must be Modern -> Ancestral

FIMO calculates the score as REF-ALT. Since I want to get positive values as one where the modern allele is binding more than archaic allele, I set the variant as alt (=modern) -> ref (=ancestral). In the FIMO config file I set assembly_type=ref

In [11]:
# Split entries on semicolon and explode
split_MPRA_results = MPRA_results["variants_genomic"].fillna("").astype(str).str.split(";").explode().str.strip()

# Regex to capture chr, pos, ref, alt
pat = re.compile(r"^(chr[\w]+):(\d+)\(([A-Za-z]+)->([A-Za-z]+)\)$")
parsed = split_MPRA_results.str.extract(pat)
parsed.columns = ["chr", "pos", "ape", "human"]

# Drop rows where parsing failed. Then drop duplicates
parsed = (
    parsed
    .dropna(subset=["chr", "pos", "ape", "human"])
    .drop_duplicates()
)
# Cast pos to int
parsed["pos"] = parsed["pos"].astype(int)

# Build output DataFrame
FIMO_input = pd.DataFrame({
    "chr": parsed["chr"],
    "start": parsed["pos"]-1,
    "end": parsed["pos"],  # one base ahead for SNVs
    "variant": parsed["human"] + "->" + parsed["ape"]
}).reset_index(drop=True)

print(FIMO_input)


          chr      start        end variant
0        chr8   30170538   30170539    C->T
1       chr11   26270252   26270253    G->A
2       chr11   26270338   26270339    T->C
3       chr11   26270395   26270396    C->A
4       chr19   42985150   42985151    G->A
...       ...        ...        ...     ...
546669   chr3  135890402  135890403    G->T
546670   chr3  135890426  135890427    A->G
546671   chr4  182061472  182061473    C->T
546672   chr4  182061556  182061557    C->A
546673   chr1  168061755  168061756    G->A

[546674 rows x 4 columns]


In [12]:
FIMO_input.to_csv("humanMPRA/TF_analysis/final/variant_tool_input/hMPRA_variant_tool_input_chonds.bed", sep='\t', header=False, index=False)